# K-Means Clustering of NYC Public School and MTA Subway Data

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [43]:
subway_df = pd.read_csv('../cleaned_data/MTA_Subway_Grouped_Data.csv')

subway_df.head()

,line,num_stations,station_census_tracts,station_zip_codes,station_boroughs,stop_names,mta_performance_data
0,1,38,"[285, 283, 283, 309, 303, 293, 283, 277, 269, ...","[10463, 10463, 10463, 10468, 10034, 10034, 100...","['Bx', 'Bx', 'Bx', 'M', 'M', 'M', 'M', 'M', 'M...","['Van Cortlandt Park-242 St', '238 St', '231 S...","[['2015-01-01', 76.2337806, 1], ['2015-02-01',..."
1,2,49,"[183, 159, 113, 101, 71, 21, 21, 15, 7, 5, 11,...","[10025, 10023, 10018, 10119, 10011, 10007, 102...","['M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', ...","['96 St', '72 St', 'Times Sq-42 St', '34 St-Pe...","[['2015-01-01', 47.2874494, 1], ['2015-02-01',..."
2,3,34,"[183, 159, 113, 101, 71, 21, 21, 15, 7, 5, 11,...","[10025, 10023, 10018, 10119, 10011, 10007, 102...","['M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', ...","['96 St', '72 St', 'Times Sq-42 St', '34 St-Pe...","[['2015-01-01', 68.6321563, 1], ['2015-02-01',..."
3,4,28,"[37, 35, 213, 351, 431, 419, 409, 403, 401, 23...","[11217, 11217, 11238, 11213, 10467, 10467, 104...","['Bk', 'Bk', 'Bk', 'Bk', 'Bx', 'Bx', 'Bx', 'Bx...","['Nevins St', 'Atlantic Av-Barclays Ctr', 'Fra...","[['2015-01-01', 48.4323757, 1], ['2015-02-01',..."
4,5,45,"[37, 35, 213, 319, 329, 804, 820, 826, 830, 78...","[11217, 11217, 11238, 11225, 11225, 11225, 112...","['Bk', 'Bk', 'Bk', 'Bk', 'Bk', 'Bk', 'Bk', 'Bk...","['Nevins St', 'Atlantic Av-Barclays Ctr', 'Fra...","[['2015-01-01', 49.3452473, 1], ['2015-02-01',..."


In [44]:
schools_df = pd.read_csv('../cleaned_data/AllBoroughs_yearly_ratings.csv')
schools_df.head(10)

,School Building Number,QR Score,Assessment Time,Location Name,Primary Address,City,Zip,Census Tract,Community District
0,K544,6,2006-04-12,NaN,NaN,NaN,NaN,NaN,NaN
1,K544,6,2007-04-28,NaN,NaN,NaN,NaN,NaN,NaN
2,K544,7,2008-03-15,NaN,NaN,NaN,NaN,NaN,NaN
3,K544,6,2010-02-10,NaN,NaN,NaN,NaN,NaN,NaN
4,K544,0,2014-12-03,NaN,NaN,NaN,NaN,NaN,NaN
5,K337,6,2006-04-26,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600.0,313.0
6,K337,1,2007-05-23,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600.0,313.0
7,K337,7,2008-05-15,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600.0,313.0
8,K337,7,2009-05-02,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600.0,313.0
9,K337,5,2012-03-28,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600.0,313.0


# Data Preparation

In [45]:
schools_df['Census Tract'] = schools_df['Census Tract'].astype(float).astype('Int64').astype(str)
schools_df.head(10)

,School Building Number,QR Score,Assessment Time,Location Name,Primary Address,City,Zip,Census Tract,Community District
0,K544,6,2006-04-12,NaN,NaN,NaN,NaN,<NA>,NaN
1,K544,6,2007-04-28,NaN,NaN,NaN,NaN,<NA>,NaN
2,K544,7,2008-03-15,NaN,NaN,NaN,NaN,<NA>,NaN
3,K544,6,2010-02-10,NaN,NaN,NaN,NaN,<NA>,NaN
4,K544,0,2014-12-03,NaN,NaN,NaN,NaN,<NA>,NaN
5,K337,6,2006-04-26,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600,313.0
6,K337,1,2007-05-23,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600,313.0
7,K337,7,2008-05-15,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600,313.0
8,K337,7,2009-05-02,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600,313.0
9,K337,5,2012-03-28,International High School at Lafayette,2630 BENSON AVENUE,BROOKLYN,11214.0,30600,313.0


# Goal:
- Use train performance as a predictor of school performance

# Troubleshooting:
- Census Tracts do not match up between subway stations and schools; use zip codes instead for mapping

In [46]:
schools_dim = schools_df[['Census Tract', 'Zip', 'Assessment Time', 'QR Score']].copy()

schools_dim = schools_dim[schools_dim['Census Tract'] != '<NA>']

schools_dim.head()

,Census Tract,Zip,Assessment Time,QR Score
5,30600,11214.0,2006-04-26,6
6,30600,11214.0,2007-05-23,1
7,30600,11214.0,2008-05-15,7
8,30600,11214.0,2009-05-02,7
9,30600,11214.0,2012-03-28,5


In [47]:
subway_df.head() # 26 rows

,line,num_stations,station_census_tracts,station_zip_codes,station_boroughs,stop_names,mta_performance_data
0,1,38,"[285, 283, 283, 309, 303, 293, 283, 277, 269, ...","[10463, 10463, 10463, 10468, 10034, 10034, 100...","['Bx', 'Bx', 'Bx', 'M', 'M', 'M', 'M', 'M', 'M...","['Van Cortlandt Park-242 St', '238 St', '231 S...","[['2015-01-01', 76.2337806, 1], ['2015-02-01',..."
1,2,49,"[183, 159, 113, 101, 71, 21, 21, 15, 7, 5, 11,...","[10025, 10023, 10018, 10119, 10011, 10007, 102...","['M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', ...","['96 St', '72 St', 'Times Sq-42 St', '34 St-Pe...","[['2015-01-01', 47.2874494, 1], ['2015-02-01',..."
2,3,34,"[183, 159, 113, 101, 71, 21, 21, 15, 7, 5, 11,...","[10025, 10023, 10018, 10119, 10011, 10007, 102...","['M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', ...","['96 St', '72 St', 'Times Sq-42 St', '34 St-Pe...","[['2015-01-01', 68.6321563, 1], ['2015-02-01',..."
3,4,28,"[37, 35, 213, 351, 431, 419, 409, 403, 401, 23...","[11217, 11217, 11238, 11213, 10467, 10467, 104...","['Bk', 'Bk', 'Bk', 'Bk', 'Bx', 'Bx', 'Bx', 'Bx...","['Nevins St', 'Atlantic Av-Barclays Ctr', 'Fra...","[['2015-01-01', 48.4323757, 1], ['2015-02-01',..."
4,5,45,"[37, 35, 213, 319, 329, 804, 820, 826, 830, 78...","[11217, 11217, 11238, 11225, 11225, 11225, 112...","['Bk', 'Bk', 'Bk', 'Bk', 'Bk', 'Bk', 'Bk', 'Bk...","['Nevins St', 'Atlantic Av-Barclays Ctr', 'Fra...","[['2015-01-01', 49.3452473, 1], ['2015-02-01',..."


In [51]:
schools_dim['Zip'] = schools_dim['Zip'].astype(float).astype('Int64').astype(str)
schools_dim.head()

,Census Tract,Zip,Assessment Time,QR Score
5,30600,11214,2006-04-26,6
6,30600,11214,2007-05-23,1
7,30600,11214,2008-05-15,7
8,30600,11214,2009-05-02,7
9,30600,11214,2012-03-28,5


In [74]:
# Creating a function to get the nearest subway station for each school based on zip code
def get_nearest_subway_station(school_zip):
    '''
    * Input: 
        - school_zip (str) - The zip code of the school
    * Output: 
        - nearest_stations (list) - List of nearest subway station by exact zip code
    '''
    nearest_stations = []
    try:
        school_zip_int = int(school_zip)
    except (ValueError, TypeError):
        return nearest_stations  # Return empty list if zip is invalid
    
    for index, row in subway_df.iterrows():
        # Convert stringified list to actual list
        station_zip_codes = eval(row['station_zip_codes']) if isinstance(row['station_zip_codes'], str) else row['station_zip_codes']
        stop_names = eval(row['stop_names']) if isinstance(row['stop_names'], str) else row['stop_names']
        
        for indx, zip_code in enumerate(station_zip_codes):
            try:
                zip_code_int = int(zip_code)
                # Check if zip code is within 2 zip codes away
                #if abs(school_zip_int - zip_code_int) <= 1:
                if school_zip_int == zip_code_int:
                    nearest_stations.append(stop_names[indx])
            except (ValueError, TypeError):
                continue
    print(len(nearest_stations))
    return nearest_stations

schools_dim['Nearest_Subway_Stations'] = schools_dim['Zip'].apply(get_nearest_subway_station)

schools_dim[10:30]

4
4
4
4
4
4
4
4
6
6
6
6
6
10
10
10
10
10
10
0
0
0
0
21
21
21
21
21
21
21
4
4
4
4
4
4
4
4
4
4
4
2
2
2
2
2
2
2
5
5
5
5
21
21
21
21
21
21
21
21
2
2
2
2
2
2
4
10
10
10
10
10
10
6
6
6
6
6
2
2
2
2
2
21
21
21
21
2
2
2
2
2
2
2
2
2
2
2
21
21
21
21
21
4
4
4
4
4
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
13
13
13
13
4
4
4
4
4
4
0
0
0
0
0
0
0
0
0
0
0
0
5
5
5
2
2
2
2
2
2
2
0
0
0
0
0
13
13
13
13
13
10
10
10
10
10
10
0
0
0
0
0
0
0
0
0
0
0
0
0
4
4
4
4
2
2
2
2
2
2
2
2
2
6
6
6
6
6
0
0
0
0
0
4
4
4
4
4
2
2
2
2
2
13
13
13
13
13
0
0
0
0
0
6
6
6
6
6
5
5
5
5
5
5
5
0
0
0
0
5
5
5
5
5
5
2
2
2
2
2
13
13
13
13
13
13
0
0
0
0
0
0
2
2
2
2
2
2
2
5
0
0
0
0
0
6
6
6
6
2
2
2
2
2
2
0
0
0
0
0
0
0
13
13
13
13
13
13
5
5
5
5
5
13
13
13
13
13
0
0
0
0
0
0
0
0
2
2
2
2
2
2
4
4
4
4
4
4
6
6
6
6
6
2
2
2
2
2
2
0
0
0
0
0
0
0
2
2
2
2
2
2
2
0
0
0
0
0
0
4
0
0
0
0
0
0
2
2
2
2
2
2
2
10
10
10
10
10
10
10
4
4
4
4
4
4
2
2
2
2
2
13
13
13
13
13
13
2
2
2
2
2
2
5
5
5
5
5
5
5
5
5
5
5
5
2
2
13
13
13
2
2
2
2
2
2
0
0
0
0
0
0
2
2
2
2
10
10
10
10
10
1

,Census Tract,Zip,Assessment Time,QR Score,Nearest_Subway_Stations
15,52900,11211,2008-05-16,7,"[Metropolitan Av, Hewes St, Lorimer St, Graham..."
16,52900,11211,2015-04-23,0,"[Metropolitan Av, Hewes St, Lorimer St, Graham..."
17,52900,11211,2019-04-06,0,"[Metropolitan Av, Hewes St, Lorimer St, Graham..."
18,21300,11225,2006-04-28,6,"[President St-Medgar Evers College, Sterling S..."
19,21300,11225,2007-05-24,1,"[President St-Medgar Evers College, Sterling S..."
20,21300,11225,2008-05-31,7,"[President St-Medgar Evers College, Sterling S..."
21,21300,11225,2011-11-16,6,"[President St-Medgar Evers College, Sterling S..."
22,21300,11225,2015-04-16,0,"[President St-Medgar Evers College, Sterling S..."
23,21300,11225,2018-11-17,0,"[President St-Medgar Evers College, Sterling S..."
37,81000,11203,2006-05-04,6,[]
